# Export Llama 3.2 1B Instruct to ExecuTorch (.pte), v2

Run the cells **one at a time, top to bottom** (Shift+Enter). Use a **CPU** runtime.
Colab ships Python 3.13, but ExecuTorch 1.0 needs **3.10–3.12**, so cell 2 creates a Python 3.12
environment at `/content/venv` and every later step uses it.

Output: `llama3_2_1b_instruct_8da4w_xnn.pte` (about 860 MB) plus `tokenizer.model`, copied to Google Drive in `gli-models/llama/`.

## 1. Check memory

In [ ]:
!free -g; python --version; df -h /content | tail -1

## 2. Python 3.12 environment + ExecuTorch from source (about 25–40 min)

In [ ]:
%%bash
set -e
cd /content
pip install -q uv
[ -x /content/venv/bin/pip ] || { rm -rf /content/venv; uv venv -q --seed -p 3.12 /content/venv; }
source /content/venv/bin/activate
uv pip install -q pip "cmake>=3.29" ninja huggingface_hub
python --version
if [ ! -d executorch ]; then
  git clone -q -b release/1.0 --recurse-submodules https://github.com/pytorch/executorch.git
fi
cd executorch
git submodule sync -q && git submodule update --init --recursive -q
export CMAKE_BUILD_PARALLEL_LEVEL=2 MAX_JOBS=2
./install_executorch.sh > /content/install.log 2>&1 || { tail -40 /content/install.log; exit 1; }
./examples/models/llama/install_requirements.sh >> /content/install.log 2>&1 || { tail -40 /content/install.log; exit 1; }
python -c "import executorch, torch; print('ExecuTorch OK, torch', torch.__version__)"


## 3. Log in to Hugging Face
Run the cell, paste your `hf_...` **Read** token at the prompt, press Enter. (The token is saved for the next cells.)

In [ ]:
from huggingface_hub import login
login()

## 4. Download Llama 3.2 1B Instruct (Meta format, about 2.5 GB)

In [ ]:
%%bash
set -e
source /content/venv/bin/activate
OUT=/content/gli-models/llama
mkdir -p $OUT
hf download meta-llama/Llama-3.2-1B-Instruct --include "original/*" --local-dir $OUT/hf
cp $OUT/hf/original/tokenizer.model $OUT/tokenizer.model
ls -lh $OUT/hf/original

## 5. Export (8da4w, group 32, 4-bit embeddings, KV cache + SDPA, XNNPACK), about 10–20 min

In [ ]:
%%bash
set -e
source /content/venv/bin/activate
OUT=/content/gli-models/llama
CK=$OUT/hf/original
NAME=llama3_2_1b_instruct_8da4w_xnn
cat > $OUT/export_config.yaml <<YAML
base:
  model_class: llama3_2
  checkpoint: $CK/consolidated.00.pth
  params: $CK/params.json
  metadata: '{"get_bos_id":128000, "get_eos_ids":[128009, 128001]}'
model:
  use_kv_cache: True
  use_sdpa_with_kv_cache: True
export:
  max_seq_length: 1024
  max_context_length: 1024
  output_dir: $OUT
  output_name: $NAME.pte
quantization:
  qmode: 8da4w
  group_size: 32
  embedding_quantize: 4,32
backend:
  xnnpack:
    enabled: True
    extended_ops: True
YAML
cd /content/executorch
START=$(date +%s)
python -m extension.llm.export.export_llm --config $OUT/export_config.yaml > /content/export.log 2>&1 || { tail -40 /content/export.log; exit 1; }
if [ ! -f $OUT/$NAME.pte ]; then
  NEW=$(find /content/executorch $OUT -name "*.pte" -newermt "@$START" | head -1)
  [ -n "$NEW" ] && mv "$NEW" $OUT/$NAME.pte
fi
ls -lh $OUT/*.pte $OUT/tokenizer.model

## 6. Check the .pte is complete (should print Loaded OK and about 0.86 GB)

In [ ]:
%%bash
source /content/venv/bin/activate
python - <<'PY'
import os
from executorch.runtime import Runtime
pte = "/content/gli-models/llama/llama3_2_1b_instruct_8da4w_xnn.pte"
print("size: %.2f GB" % (os.path.getsize(pte) / 1e9))
p = Runtime.get().load_program(pte)
print("Loaded OK:", list(p.method_names)[:6])
PY

## 7. Save to Google Drive
Authorise Drive when asked. Files go to **My Drive → gli-models → llama**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/gli-models/llama
!cp -v /content/gli-models/llama/llama3_2_1b_instruct_8da4w_xnn.pte /content/gli-models/llama/tokenizer.model /content/gli-models/llama/export_config.yaml /content/drive/MyDrive/gli-models/llama/
!ls -lh /content/drive/MyDrive/gli-models/llama/